# Gradient fields

In this tutorial we will see how we can chain together the interpolation and deposition methods in `smudgy`. 

+ First, we will compute the gradient of a particle field using the {py:meth}`~smudgy.pointcloud.PointCloud.interpolate` method.
+ Then, we deposit the newly computed gradient to a 2D grid and visualize it.

## Interpolating particle gradients and depositing them

First, let's load all the necessary packages and modules.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import smudgy as sm

np.random.seed(0)

In [ ]:
try:
    plt.style.use('custom')
except:
    pass
%matplotlib inline
%config InlineBackend.figure_format='retina'

Then, we initialize a (random) 2D particle cloud in a periodic box of size 1, globally set an isotropic cubic kernel and set the number of neighbors to 32. We also assign to each particle a scalar field value $u$:

In [ ]:
boxsize = 1.0
N = 1000
positions = np.random.uniform(0.0, boxsize, size=(N, 2))
weights = np.ones(N)

pc = sm.PointCloud(positions,
                   weights,
                   boxsize=boxsize).\
    global_setup(kernel_name='cubic_spline',
                 structure='isotropic',
                 num_neighbors=32)

# add field to the point cloud
particle_u = np.random.normal(loc=10, scale=1, size=N)
particle_u = np.clip(particle_u, 0.0, None)

pc.add_fields('u', particle_u)

Next, we compute smoothing lenghts and densities with the globally set parameters:

In [ ]:
pc.compute_smoothing()
pc.compute_density()

Now we can compute (interpolate) the energy gradients at the particle positions. We do so by not passing `query_positions`, which will default to use `pc.positions`:

In [ ]:
# shape = (N, 1, 2)
u_gradient = pc.interpolate(
    fields=['u'],
    mode="gradient")

We register the newly computed gradient field to the point cloud instance:

In [ ]:
pc.add_fields('u_gradient', u_gradient[:, 0, :])

Now we can access that gradient field by name to deposit it to a uniform 2D grid of 256 cells on each side:

In [ ]:
nx = 256

# shape = (2, nx, nx)
grid_gradients = pc.deposit(
    fields=['u_gradient'],
    structure='isotropic',
    averaged=True,
    gridnums=nx,
    )

Finally, we visualize the two gradient components side-by-side together with the particles shown in gray:

In [ ]:
kwargs = dict(
    extent = [0, boxsize, 0, boxsize],
    cmap = 'seismic'
)

fig, ax = plt.subplots(
    1, 2,
    figsize=(4, 2),
    sharex=True,
    sharey=True,
    constrained_layout=True,
)

im = ax[0].imshow(grid_gradients[0], alpha=1.0, **kwargs)
_  = ax[1].imshow(grid_gradients[1], alpha=1.0, **kwargs)

ax[0].scatter(positions[:, 0], positions[:, 1], s=2, alpha=1, color='gray', ec='none')
ax[1].scatter(positions[:, 0], positions[:, 1], s=2, alpha=1, color='gray', ec='none')

for a in ax:
    a.set_xlabel(r'$x$')
ax[0].set_ylabel(r'$y$')

ax[0].set_title(r'$x$-component of $\nabla u$')
ax[1].set_title(r'$y$-component of $\nabla u$')
fig.colorbar(im, ax=ax, shrink=0.8, label=r'Gradient value')

fig.savefig("gradients.png", dpi=500, bbox_inches="tight")
plt.close()

![Gradient figure](gradients.png)

:::{note}
Gradients computed with this method are different compared to gradients of grids:
- Depositing per‑particle gradients preserves the local gradient estimates computed at particle scales and then averages them into cells.
- Computing the gradient of the deposited scalar field (e.g. via finite‑differences) produces a different result because of cell averaging and the finite‑difference operator, i.e. 

    $$ 
    \mathrm{deposit} (\nabla f) \neq \nabla(\mathrm{deposit}(f)).
    $$

Use the first for particle‑scale gradient estimates and the second when you prefer derivatives computed on the (possibly smoothed) grid representation.
:::